# 02 · Build the Desi Fit try-on dataset (CPU runtime is enough)
Runs the whole dataset pipeline on the images already in Drive:

1. **catalog + view classification**: every image labelled flat / flat_lower / detail / mff (model front full-length) / mo (other model view). ~16.8k images ≈ 1–1.5 h. Resumable.
2. **gold check**: classifier accuracy on the 137 hand-labelled images.
3. **candidate pairs**: a **worn** pair for every eligible SKU (garment = a different front photo of the same outfit; the primary type), plus a **flat** pair where ETHNC has a flat lay.
4. **review pages**: HTML pages you open in a browser and work through with the keyboard.
5. **export** of the accepted pairs to VITON-HD / CatVTON format with a leak-free split.

Then run **03_gpu_masks** on a T4 (masks, DensePose, garment cut-outs) and **04_compare_garment_inputs** to choose which garment image to train with.

> Runtime → Change runtime type → **CPU** is fine for this notebook. Keep your GPU quota for 03.

## 0 · Setup
Mount Drive and get the code. Put **either** your GitHub URL in `REPO_URL`, **or** leave it empty to use `desi-fit-ai.zip` from the dataset folder on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
ROOT   = '/content/drive/MyDrive/AainaClosetDataset'      # scraped images live here
BUILD  = f'{ROOT}/_build'                                  # all intermediate CSVs + review pages
DATASET= '/content/drive/MyDrive/DesiFitAI/processed/DesiFit-VTON-v1'   # final exported dataset
REPO_URL = ''   # e.g. 'https://github.com/<you>/desi-fit-ai.git'  (private repo: use a token URL)
os.makedirs(BUILD, exist_ok=True)

if REPO_URL:
    !rm -rf /content/desi-fit-ai && git clone -q $REPO_URL /content/desi-fit-ai
else:
    !rm -rf /content/desi-fit-ai && unzip -q "$ROOT/desi-fit-ai.zip" -d /content/
%cd /content/desi-fit-ai
!git log -1 --oneline 2>/dev/null || ls

### Python environment
`mediapipe==0.10.14` is the last release that ships its pose/face models inside the package (no downloads). It has wheels up to Python 3.12. If Colab has moved to a newer Python, this cell builds a 3.11 virtualenv with `uv` and every later step runs through `PY`.

In [ ]:
import sys
PY = sys.executable
if sys.version_info >= (3, 13):
    !pip -q install uv
    !uv venv -q -p 3.11 /content/venv311
    PY = '/content/venv311/bin/python'
    !uv pip -q install --python $PY -r requirements-data.txt
else:
    !pip -q install -r requirements-data.txt
!$PY -c "import mediapipe, cv2, pandas; print('mediapipe', mediapipe.__version__)"

## 1 · Catalog + view classification
Smoke test on 60 images first; then the full run. If Colab disconnects, just re-run the cell — finished images are skipped.

In [ ]:
!$PY -m data_tools.classify_all --root "$ROOT" --out "$BUILD/views_smoke.csv" --limit 60 --workers 2

In [ ]:
!$PY -m data_tools.classify_all --root "$ROOT" --out "$BUILD/views.csv" --workers 2 --chunk 300

## 2 · Gold-set check
Expect roughly: accuracy ≈ 0.88, flat recall/precision 1.0, target pick 21/22. If these numbers drop a lot, something changed in the images or the environment — stop and look before building pairs.

In [ ]:
!$PY -m data_tools.gold --views "$BUILD/views.csv"

## 3 · Candidate pairs + statistics

In [ ]:
!$PY -m data_tools.pairs --catalog "$BUILD/catalog.csv" --views "$BUILD/views.csv" --out "$BUILD/pairs_candidates.csv"
!$PY -m data_tools.stats --build "$BUILD" > "$BUILD/stats_before_review.md"
from IPython.display import Markdown; Markdown(open(f'{BUILD}/stats_before_review.md').read())

## 4 · Review pages
Writes `review_001.html`, `review_002.html`, … (60 pairs each) to `_build/review/`.

**How to review** (see `docs/review_guide.md`):
1. In Drive, download a page and open it in Chrome/Edge on your laptop (it is self-contained, works offline).
2. `A` accept · `R` then `1-6` reject with a reason · click a thumbnail to make it the PERSON image · Shift+click to make it the CLOTH image · `J`/`K` move.
3. Click **Download decisions** and upload the CSV to `_build/review_csv/`.
4. **Both of you review page 001 independently** (save as `review_001_maira.csv` and `review_001_mihab.csv`) → agreement (Cohen's κ) for the report. Split the remaining pages.

In [ ]:
!$PY -m data_tools.review --pairs "$BUILD/pairs_candidates.csv" --catalog "$BUILD/catalog.csv" --root "$ROOT" --out-dir "$BUILD/review" --per-page 60

### After reviewing: agreement + merge

In [ ]:
import glob
os.makedirs(f'{BUILD}/review_csv', exist_ok=True)
pair = [f'{BUILD}/review_csv/review_001_maira.csv', f'{BUILD}/review_csv/review_001_mihab.csv']
if all(os.path.exists(p) for p in pair):
    !$PY -m data_tools.review --pairs x --catalog x --root x --out-dir x --kappa {pair[0]} {pair[1]}
# merge: for page 001 keep one of the two copies (here: maira's) so each pair has exactly one decision
!mkdir -p "$BUILD/review_final" && cp "$BUILD"/review_csv/*.csv "$BUILD/review_final/" && rm -f "$BUILD/review_final/review_001_mihab.csv"
!$PY -m data_tools.review --pairs "$BUILD/pairs_candidates.csv" --catalog x --root x --out-dir x --merge "$BUILD/review_final/*.csv"

## 5 · Export (VITON-HD / CatVTON layout, 768×1024, split by design group)

In [ ]:
!$PY -m data_tools.export --pairs "$BUILD/pairs_reviewed.csv" --root "$ROOT" --out "$DATASET" --worn-cloth faceless   # change after notebook 04 if another variant wins
!$PY -m data_tools.stats --build "$BUILD" --export "$DATASET" > "$BUILD/stats_after_export.md"
Markdown(open(f'{BUILD}/stats_after_export.md').read())

### Look at what you exported
Never train on data you have not looked at.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
p = pd.read_csv(f'{DATASET}/pairs.csv'); p = p[p.status=='ok'].sample(min(8, len(p)), random_state=0)
fig, ax = plt.subplots(len(p), 3, figsize=(9, 4*len(p)))
for i, r in enumerate(p.itertuples()):
    d = f'{DATASET}/{r.split}'
    for j, sub in enumerate(['image', 'cloth', 'cloth-mask']):
        ax[i, j].imshow(Image.open(f'{d}/{sub}/{r.pair_id}.jpg')); ax[i, j].axis('off')
    ax[i, 0].set_title(f'{r.pair_id} {r.brand} {r.category}', fontsize=8)
plt.tight_layout()